# 02 · Store-level forecast and accuracy
**Business question:** how well would a simple baseline forecast perform for each store, and how do we measure it the way planners do?

Builds the planner's toolkit at store × week level: share of total, week-over-week change, moving average, ranking, running totals, a **naive forecast** (this week = last week) and its **WMAPE** and **bias**.

*IBP comparison: these are the same key figures as a demand planning accuracy dashboard (lag forecast vs actuals, WMAPE, bias), built by hand in SQL.*

> **Run one notebook at a time.** DuckDB allows only one open connection to `practice.duckdb`. Before opening another notebook, use *Kernel → Shut Down Kernel* in this one.

In [ ]:
import duckdb

# Connect to the local DuckDB file built in 00_setup_and_load
con = duckdb.connect('../data/practice.duckdb')

# Memory settings: needed for the 59M-row sales_daily table on a laptop
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

## 1. Store share of total units (one week)

In [ ]:
# Each store's share of all-store units in week 11101
# Window function keeps every store row and adds the all-store total next to it
con.sql("""
SELECT store_id,
       wm_yr_wk,
       units,
       SUM(units) OVER (PARTITION BY wm_yr_wk)     AS units_sold_all_stores,
       ROUND(units / units_sold_all_stores * 100, 1) AS pct_units_sold_bystore
FROM weekly_store_sales
WHERE wm_yr_wk = 11101
ORDER BY units DESC
""").df()

,store_id,wm_yr_wk,units,units_sold_all_stores,pct_units_sold_bystore
0,CA_3,11101,30472.0,189942.0,16.0
1,WI_3,11101,25972.0,189942.0,13.7
2,CA_1,11101,23715.0,189942.0,12.5
3,TX_2,11101,21993.0,189942.0,11.6
4,CA_2,11101,17844.0,189942.0,9.4
5,TX_3,11101,16936.0,189942.0,8.9
6,TX_1,11101,15442.0,189942.0,8.1
7,WI_2,11101,14369.0,189942.0,7.6
8,WI_1,11101,12577.0,189942.0,6.6
9,CA_4,11101,10622.0,189942.0,5.6


## 2. Week-over-week change (LAG)

In [ ]:
# LAG pulls the previous week's value onto the current row, per store
con.sql("""
SELECT store_id,
       wm_yr_wk,
       units                                                     AS current_wk_units,
       LAG(units) OVER (PARTITION BY store_id ORDER BY wm_yr_wk) AS previous_wk_units,
       units - LAG(units) OVER (PARTITION BY store_id ORDER BY wm_yr_wk) AS change_units_curr_previous_wk
FROM weekly_store_sales
WHERE store_id = 'CA_1'
ORDER BY wm_yr_wk
""").df()

,store_id,wm_yr_wk,current_wk_units,previous_wk_units,change_units_curr_previous_wk
0,CA_1,11101,23715.0,NaN,NaN
1,CA_1,11102,24645.0,23715.0,930.0
2,CA_1,11103,21153.0,24645.0,-3492.0
3,CA_1,11104,21716.0,21153.0,563.0
4,CA_1,11105,20167.0,21716.0,-1549.0
...,...,...,...,...,...
273,CA_1,11613,31771.0,31347.0,424.0
274,CA_1,11614,34639.0,31771.0,2868.0
275,CA_1,11615,33226.0,34639.0,-1413.0
276,CA_1,11616,34033.0,33226.0,807.0


## 3. Four-week moving average forecast

In [ ]:
# Forecast = average of the 4 weeks BEFORE the current week
# "4 PRECEDING AND 1 PRECEDING" excludes the current week, so the forecast never sees its own actual
con.sql("""
SELECT store_id,
       wm_yr_wk,
       units AS actuals,
       AVG(units) OVER (PARTITION BY store_id
                        ORDER BY wm_yr_wk
                        ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING) AS avg_forecast_prev4
FROM weekly_store_sales
WHERE store_id = 'CA_3'
ORDER BY wm_yr_wk
""").df()

,store_id,wm_yr_wk,actuals,avg_forecast_prev4
0,CA_3,11101,30472.0,NaN
1,CA_3,11102,33726.0,30472.00
2,CA_3,11103,28088.0,32099.00
3,CA_3,11104,27939.0,30762.00
4,CA_3,11105,28246.0,30056.25
...,...,...,...,...
273,CA_3,11613,42573.0,44365.00
274,CA_3,11614,45825.0,44044.50
275,CA_3,11615,43976.0,44011.00
276,CA_3,11616,42909.0,43802.00


## 4. Ranking and running totals

In [ ]:
# Rank stores by revenue within one week
con.sql("""
SELECT store_id,
       wm_yr_wk,
       revenue,
       RANK() OVER (ORDER BY revenue DESC) AS rank
FROM weekly_store_sales
WHERE wm_yr_wk = 11110
ORDER BY rank
""").df()

,store_id,wm_yr_wk,revenue,rank
0,CA_3,11110,86534.82,1
1,CA_1,11110,64769.65,2
2,WI_3,11110,59443.00,3
3,TX_2,11110,56593.05,4
4,CA_2,11110,50902.59,5
5,TX_3,11110,43116.46,6
6,TX_1,11110,42188.02,7
7,WI_2,11110,31273.37,8
8,WI_1,11110,30968.30,9
9,CA_4,11110,30202.79,10


In [ ]:
# Cumulative units for TX_1 over the first 13 weeks
con.sql("""
SELECT store_id,
       wm_yr_wk,
       units,
       SUM(units) OVER (PARTITION BY store_id
                        ORDER BY wm_yr_wk
                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_units_stores
FROM weekly_store_sales
WHERE wm_yr_wk BETWEEN 11101 AND 11113
  AND store_id = 'TX_1'
ORDER BY wm_yr_wk
""").df()

,store_id,wm_yr_wk,units,cum_units_stores
0,TX_1,11101,15442.0,15442.0
1,TX_1,11102,15788.0,31230.0
2,TX_1,11103,16248.0,47478.0
3,TX_1,11104,14819.0,62297.0
4,TX_1,11105,14934.0,77231.0
5,TX_1,11106,15765.0,92996.0
6,TX_1,11107,16653.0,109649.0
7,TX_1,11108,14645.0,124294.0
8,TX_1,11109,14893.0,139187.0
9,TX_1,11110,16022.0,155209.0


## 5. Naive forecast and forecast error
**Naive forecast:** this week's forecast = last week's actual. It is the baseline any real forecast (statistical or ML) must beat.

In [ ]:
# Naive forecast and error for CA_2
# Error here = actual − forecast (positive = we under-forecast)
con.sql("""
SELECT store_id,
       wm_yr_wk,
       units,
       LAG(units) OVER (PARTITION BY store_id ORDER BY wm_yr_wk)         AS naive_forecast,
       units - LAG(units) OVER (PARTITION BY store_id ORDER BY wm_yr_wk) AS forecast_error
FROM weekly_store_sales
WHERE store_id = 'CA_2'
ORDER BY wm_yr_wk
""").df()

,store_id,wm_yr_wk,units,naive_forecast,forecast_error
0,CA_2,11101,17844.0,NaN,NaN
1,CA_2,11102,17948.0,17844.0,104.0
2,CA_2,11103,16185.0,17948.0,-1763.0
3,CA_2,11104,16268.0,16185.0,83.0
4,CA_2,11105,16012.0,16268.0,-256.0
...,...,...,...,...,...
273,CA_2,11613,31080.0,30024.0,1056.0
274,CA_2,11614,33708.0,31080.0,2628.0
275,CA_2,11615,32369.0,33708.0,-1339.0
276,CA_2,11616,33995.0,32369.0,1626.0


## 6. WMAPE and bias of the naive forecast, by store
- **WMAPE** = Σ |forecast − actual| ÷ Σ actual. Weighted by volume, so big weeks count more than small ones.
- **Bias** = Σ (forecast − actual) ÷ Σ actual. Negative = under-forecasting, positive = over-forecasting.

A CTE is needed because the error comes from a window function (LAG), and a window result cannot sit inside SUM directly.

In [ ]:
con.sql("""
WITH naive_fcst_error AS (
    -- Step 1: naive forecast per store-week
    SELECT store_id,
           wm_yr_wk,
           units,
           LAG(units) OVER (PARTITION BY store_id ORDER BY wm_yr_wk) AS naive_fcst
    FROM weekly_store_sales
)
-- Step 2: aggregate the errors per store
SELECT store_id,
       SUM(units)      AS agg_actuals,
       SUM(naive_fcst) AS agg_naive_FCST,
       ROUND(SUM(ABS(naive_fcst - units)) / SUM(units) * 100, 1) AS wmape,
       ROUND(SUM(naive_fcst - units)      / SUM(units) * 100, 1) AS bias
FROM naive_fcst_error
WHERE naive_fcst IS NOT NULL                -- first week has no forecast
  AND wm_yr_wk != 11617                     -- last week is partial (2 days)
GROUP BY store_id
ORDER BY wmape DESC
""").df()

,store_id,agg_actuals,agg_naive_FCST,wmape,bias
0,WI_2,6673338.0,6648737.0,13.8,-0.4
1,WI_3,6507309.0,6503718.0,11.0,-0.1
2,TX_1,5668590.0,5658668.0,6.3,-0.2
3,TX_3,6179318.0,6166935.0,6.2,-0.2
4,CA_3,11317851.0,11305414.0,6.2,-0.1
5,CA_1,7796480.0,7786162.0,5.9,-0.1
6,TX_2,7297630.0,7290253.0,5.5,-0.1
7,WI_1,5238873.0,5223074.0,5.2,-0.3
8,CA_4,4165169.0,4155711.0,4.2,-0.2
9,CA_2,5786817.0,5770666.0,4.1,-0.3


**Findings: naive forecast accuracy**
- Store-level WMAPE ranges from **4.1% (CA_2) to 13.8% (WI_2)**. At store × week level, demand is smooth, so even a naive forecast does well. Accuracy will be much worse at item level.
- WI_2 and WI_3 are clear outliers (13.8% and 11.0%) and would be the first stores to investigate.
- Bias is slightly **negative in every store (−0.1% to −0.4%)**: sales grew over the period, so "last week" always lags a little behind. A trend-aware forecast should remove this.
- Excluding the partial final week matters: without the filter, its 2-day "drop" would add a large fake error to every store.